# Station Stacking v11 9AM Settlement - KDAL

Experimental notebook for `KDAL`.

This controlled rerun keeps the exact v11 feature/model contract, uses only point-in-time-safe 9 AM forecast and observation data, requires Wunderground station-history daily-high labels with no fallback, and writes separate artifacts to `data/calibration/station_stacking_v11_9AM_settlement`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KDAL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_9am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v11_9AM_wunderground_settlement_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v11"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
3,KDAL,gfs,2007,2021-01-01,2026-06-30
4,KDAL,hrrr,2007,2021-01-01,2026-06-30
5,KDAL,nbm,1991,2021-01-01,2026-06-30


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11",
    target_mode="remaining_warmup",
    target_source="wunderground_only",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11_9AM_settlement",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_9AM_settlement/KDAL_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-07-13 10:09:06,533] Using an existing study with name 'KDAL_v11_remaining_warmup_base_xgboost_mae_f_wide' instead of creating a new one.
[I 2026-07-13 10:09:06,730] Using an existing study with name 'KDAL_v11_remaining_warmup_base_lightgbm_mae_f_wide' instead of creating a new one.
[I 2026-07-13 10:09:06,943] Using an existing study with name 'KDAL_v11_remaining_warmup_base_catboost_mae_f_wide' instead of creating a new one.
[I 2026-07-13 10:20:44,143] Trial 2 finished with value: 2.0203237772411025 and parameters: {'iterations': 2931, 'learning_rate': 0.001783588943232587, 'depth': 10, 'l2_leaf_reg': 0.9942958882300413, 'random_strength': 7.186216756161439, 'bagging_temperature': 12.184767612363034, 'border_count': 110, 'rsm': 0.6158971964095841, 'huber_delta': 1.5}. Best is trial 0 with value: 2.0203237772411025.
[I 2026-07-13 11:37:53,278] Trial 3 finished with value: 2.4951457243547837 and parameters: {'iterations': 801, 'learning_rate': 0.19767902101650964, 'depth': 12, 'l

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,721,1.588154,2.104024
1,validation_2024_2025,lightgbm,721,1.459284,1.983076
2,validation_2024_2025,catboost,721,1.504479,2.004677
3,validation_2024_2025,provider_mean,721,2.352713,3.081034
4,validation_2024_2025,provider_median,721,2.053936,2.877243
5,validation_2024_2025,nbm_raw,721,1.910307,2.744604
6,validation_2024_2025,hrrr_raw,721,4.265447,5.095655
7,validation_2024_2025,gfs_raw,721,2.654359,3.526307
8,test_2026,xgboost,109,1.649870,2.065754
9,test_2026,lightgbm,109,1.615900,2.051264


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    target_source=config.effective_target_source,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v11_9AM_settlement",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_9AM_settlement/model_weights/KDAL_station_high_regressor_v11_9AM_wunderground_settlement_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_9AM_settlement/model_weights/KDAL_station_high_regressor_v11_9AM_wunderground_settlement_stack.json'))

## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v4_observed_precip_any,100.000000
1,v4_forecast_wet_observed_dry,100.000000
2,v4_forecast_observed_precip_match,100.000000
3,v4_all_forecast_precip,100.000000
4,v4_any_forecast_precip,100.000000
5,climatology_high_10y_std_f,100.000000
6,climatology_high_10y_count,100.000000
7,provider_mean_minus_climatology_10y_f,100.000000
8,v8_month_remaining_warmup_count,100.000000
9,v4_observed_wet_forecast_dry,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
168,v2_recent_heat_anomaly_f,numeric
169,v2_recent_heat_momentum_f,numeric
170,v2_morning_warmup_to_consensus_f,numeric
171,v2_consensus_minus_7d_actual_f,numeric
172,v2_spread_per_warmup_f,numeric
173,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,96.896897
1,observed_temp_change_last_3h_f,96.896897
2,observed_high_so_far_change_since_9am_f,96.896897
3,observed_morning_warmup_rate_f_per_hour,4.104104


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,109,66,60.550459
7,oof_2026,ridge_stack,109,65,59.633028
3,oof_2026,lightgbm,109,64,58.715596
8,oof_2026,xgboost,109,59,54.128440
4,oof_2026,nbm_raw,109,54,49.541284
6,oof_2026,provider_median,109,41,37.614679
5,oof_2026,provider_mean,109,40,36.697248
1,oof_2026,gfs_raw,109,37,33.944954
2,oof_2026,hrrr_raw,109,9,8.256881
12,validation_2024_2025,lightgbm,721,454,62.968100


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,ridge_stack,138,1.369446,1.890651,v9
1,test_2026,xgboost,138,1.402066,1.889472,v11
2,test_2026,xgboost,138,1.419940,1.895994,v9
3,test_2026,ridge_stack,138,1.429989,1.907881,v11
4,test_2026,lightgbm,138,1.454606,1.932482,v11
...,...,...,...,...,...,...
83,validation_2024_2025,hrrr_raw,660,5.477361,6.266007,v3
84,validation_2024_2025,hrrr_raw,668,5.486824,6.274688,v7
85,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v5
86,validation_2024_2025,hrrr_raw,606,5.497186,6.283967,v6


## 2026 OOF Weather Brackets


In [15]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bucket_log_loss,bracket_accuracy_pct,p95_absolute_error_f,large_miss_5f_pct
0,xgboost,109,1.649870,2.065754,1.502754,33.944954,4.110062,0.917431
1,lightgbm,109,1.615900,2.051264,1.500997,33.027523,4.497331,2.752294
2,catboost,109,1.560946,1.943482,1.442995,35.779817,3.595413,1.834862
3,ridge_stack,109,1.546526,1.963744,1.445130,35.779817,3.929993,0.917431
4,provider_mean,109,2.357465,2.951205,1.632681,26.605505,5.230031,7.339450
5,provider_median,109,2.314051,2.887820,1.672206,22.93578,5.093096,6.422018
6,nbm_raw,109,1.973103,2.500580,1.641672,33.027523,5.098018,6.422018
7,hrrr_raw,109,4.492339,5.142789,1.828980,7.33945,8.863979,40.366972
8,gfs_raw,109,2.628724,3.403932,1.953933,22.018349,6.545635,11.009174
